# 🧊 QQQ Weekly Research — Parte 3.5: Construcción del Universo Operativo
## Pipeline: Datos estructurados → Construcción del universo operativo → Generación de señales y operaciones

---

**Objetivo:** Construir el universo operativo utilizado en los análisis posteriores del proyecto, integrando las señales seleccionadas en las etapas previas.

Las señales y operaciones se derivan directamente de los datasets validados en etapas anteriores, manteniendo consistencia metodológica a lo largo del pipeline.

---

Entradas

• Dataset estructural (Parte 1)
• Variables operativas validadas (Parte 2)
• Configuración seleccionada (Parte 3)

Salidas

• Universo operativo
• Operaciones OR
• Operaciones AND
• Paneles de análisis
• Metadata del estudio

---

### 🗺️ Mapa del Notebook

| # | Bloque | Descripción |
|---|--------|-------------|
| 0 | **Setup** | Librerías, parámetros congelados y rutas del estado canónico |
| 1 | **Carga de Artefactos Validados** | Lectura de los CSV de Parte 1 y Parte 2 |
| 2 | **Lógica Operativa** | Funciones (holding, señales) |
| 3 | **Construcción del Universo Operativo** | Universo operativo sobre el sustrato validado |
| 4 | **Extracción de Operaciones** | Trade list OR/AND y paneles temporales |
| 5 | **Validación de Integridad** | Validación de integridad y consistencia |
| 6 | **Verificación de Artefactos del Modelo** | Carga del bundle si existe (transición) |
| 7 | **Persistencia de Resultados y Metadata** | Escritura de artefactos + firma del estado |
| 8 | **Validación y Aprobación del Estado** | Promoción manual `pending` → `canonical` |
| 9 | **Resumen Ejecutivo** | Resumen trazable del estado producido |
| — | **Observaciones y mejoras futuras** | Hallazgos documentados (no implementados) |

---


---
## ⚙️ BLOQUE 0 — Setup, Parámetros y Rutas

Inicializa el entorno de trabajo y consolida los parámetros operativos utilizados en esta etapa.

Además, define las rutas de entrada y salida necesarias para la generación de señales, operaciones y artefactos derivados del estudio.

In [28]:
# =========================================================
# BLOQUE 0 — SETUP Y PARÁMETROS 
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import json
import pickle
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

# ─── Datasets generados en etapas previas ──
PART1_CSV = Path("./qqq_part1_dataset.csv")
PART2_CSV = Path("./qqq_part2_validation_dataset.csv")

# ─── Universo operativo: filtro y umbrales ──
MODERN_YEAR_FILTER     = 2010    # df_modern = año >= 2010 
DD_RANK_THRESHOLD      = 0.65    # umbral absoluto del score DD+ATR
DD_HOLD_WEEKS          = 4       # holding de la señal DD+ATR
STATE_SCORE_PCT_THRESH = 0.70    # percentil de state_score_opt
STATE_HOLD_WEEKS       = 8       # holding de la señal StateOpt

# ─── Variables internas guardadas en los paneles por operación ──
INTERNAL_VARS = [
    "cluster", "meta_state", "cluster_prob", "pred_prob_bull",
    "state_score_opt", "bull_rank", "trend_rank", "drawdown", "atr_pct",
    "mom_4", "mom_13", "mom_26", "mom_52", "trend_13_52", "vol_13",
    "score_dd_atr", "dd_raw", "dd_held", "opt_raw", "opt_held",
]

# ─── Rutas de salida del universo operativo ──
CACHE_DIR = Path("./qqq_p4_cache")

# Artefactos del ESTADO OPERATIVO
STATE_ARTIFACTS = {
    "live_df":   CACHE_DIR / "live_df.pkl",
    "or_trades": CACHE_DIR / "or_trades.pkl",
    "or_panels": CACHE_DIR / "or_panels.pkl",
    "metadata":  CACHE_DIR / "metadata.pkl",
}

# Modelo de inferencia (si se encuentra disponible) 
BUNDLE_PATH = CACHE_DIR / "model_artifacts.joblib"

MANIFEST_PATH = CACHE_DIR / "MANIFEST.json"

# Versión de esta etapa.
PIPELINE_VERSION = "3.5.0"

# ─── Interruptores manuales ─────────
FORCE_REBUILD = False

# Habilita el cierre formal del pipeline tras revisión manual.
CANONIZE_PENDING_STATE = False

# Referencia de conteos esperada (opcional, segundo cinturón). Se deja en None hasta
# Se mantiene en None hasta disponer de una referencia estable.
EXPECTED_OR_TRADES  = None
EXPECTED_AND_TRADES = None

CACHE_DIR.mkdir(exist_ok=True)

print("✅ Setup completo.")
print(f"   Entrada (Parte 1) : {PART1_CSV}")
print(f"   Entrada (Parte 2) : {PART2_CSV}")
print(f"   FORCE_REBUILD     : {FORCE_REBUILD}")
print(f"   CANONIZE_PENDING  : {CANONIZE_PENDING_STATE}")


✅ Setup completo.
   Entrada (Parte 1) : qqq_part1_dataset.csv
   Entrada (Parte 2) : qqq_part2_validation_dataset.csv
   FORCE_REBUILD     : False
   CANONIZE_PENDING  : False


---
## 📥 BLOQUE 1 — Carga de Artefactos Validados (CSV de Parte 1 y Parte 2)

La Parte 3.5 utiliza los datasets generados y validados en las etapas previas del proyecto.

• qqq_part1_dataset.csv: estructura base del mercado, variables derivadas y clasificación de regímenes.

• qqq_part2_validation_dataset.csv: variables operativas, rankings y métricas seleccionadas durante la etapa de validación.

El universo operativo se construye sobre el sustrato validado de Parte 2 (donde se encuentra state_score_opt), aplicando los mismos filtros y reglas metodológicas definidos en la Parte 3.

De esta forma se mantiene consistencia metodológica entre la validación del modelo y la generación de señales operativas.


In [29]:
# =========================================================
# BLOQUE 1 — CARGA DE DATASETS DE ENTRADA
# =========================================================

def load_inputs():
    """
    Carga los datasets generados en las etapas previas del proyecto.
    No realiza transformaciones ni recalcula variables.
    """

    if not PART2_CSV.exists():
        raise FileNotFoundError(
            f"No se encontró {PART2_CSV}. "
            f"Este archivo es necesario para construir el universo operativo."
        )

    df_scores = pd.read_csv(
        PART2_CSV,
        index_col=0,
        parse_dates=True
    )

    df_base = None

    if PART1_CSV.exists():
        df_base = pd.read_csv(
            PART1_CSV,
            index_col=0,
            parse_dates=True
        )

    return df_base, df_scores


df_base, df_scores = load_inputs()

print("✅ Datasets cargados correctamente.")

if df_base is not None:
    print(
        f"   Parte 1 : {df_base.shape[0]:,} filas "
        f"({df_base.index.min().date()} → {df_base.index.max().date()})"
    )
else:
    print(
        "   Parte 1 : no disponible "
        "(solo se utilizará el dataset operativo)"
    )

print(
    f"   Parte 2 : {df_scores.shape[0]:,} filas "
    f"({df_scores.index.min().date()} → {df_scores.index.max().date()})"
)

# ─────────────────────────────────────────────────────────
# Validación mínima de columnas requeridas
# ─────────────────────────────────────────────────────────

REQUIRED_COLS = [
    "drawdown",
    "atr_pct",
    "state_score_opt",
    "returns",
    "Close",
]

missing_cols = [
    col for col in REQUIRED_COLS
    if col not in df_scores.columns
]

if missing_cols:
    raise ValueError(
        f"Faltan columnas requeridas en Parte 2: {missing_cols}"
    )

print(f"✅ Columnas requeridas presentes: {REQUIRED_COLS}")

✅ Datasets cargados correctamente.
   Parte 1 : 1,328 filas (2001-01-05 → 2026-06-12)
   Parte 2 : 354 filas (2001-01-26 → 2025-05-30)
✅ Columnas requeridas presentes: ['drawdown', 'atr_pct', 'state_score_opt', 'returns', 'Close']


---
## 🧮 BLOQUE 2 — Reglas Operativas del Modelo

Funciones encargadas de generar las señales operativas del estudio.

Las reglas, umbrales y horizontes de permanencia utilizados en esta etapa corresponden a la configuración seleccionada durante el proceso de validación y evaluación desarrollado en las etapas anteriores.

La variable state_score_opt proviene del dataset operativo generado en etapas anteriores y se utiliza como insumo para la construcción de señales.


In [30]:
# =========================================================
# BLOQUE 2 — LÓGICA OPERATIVA
# =========================================================

def build_holding_signal(base_signal, hold_weeks, index):
    """
    Construye una señal con horizonte de permanencia fijo.
    Una vez activada, la señal permanece activa durante
    `hold_weeks` períodos consecutivos.
    """
    sig = pd.Series(False, index=index)
    remaining = 0
    for i in range(len(index)):
        if remaining > 0:
            sig.iloc[i] = True
            remaining -= 1
        elif base_signal.iloc[i]:
            sig.iloc[i] = True
            remaining = hold_weeks - 1
    return sig


def build_holding_signal_rich(base_signal, hold_weeks, index):
    """
    Versión enriquecida de la señal con holding.
    Además del estado activo, devuelve información auxiliar
    para análisis operativos:
    - held: señal activa
    - is_fresh: activación nueva
    - trigger_date: fecha de activación
    - weeks_remaining: semanas restantes de permanencia
    """
    held            = pd.Series(False, index=index)
    is_fresh        = pd.Series(False, index=index)
    trigger_date    = pd.Series(pd.NaT, index=index, dtype="datetime64[ns]")
    weeks_remaining = pd.Series(0, index=index)
    remaining, last_trigger = 0, None
    for i in range(len(index)):
        if remaining > 0:
            held.iloc[i] = True
            remaining -= 1
            weeks_remaining.iloc[i] = remaining
            trigger_date.iloc[i] = last_trigger
        elif base_signal.iloc[i]:
            held.iloc[i] = True
            is_fresh.iloc[i] = True
            remaining = hold_weeks - 1
            weeks_remaining.iloc[i] = remaining
            last_trigger = index[i]
            trigger_date.iloc[i] = last_trigger
    return held, is_fresh, trigger_date, weeks_remaining


def count_events(df, col):
    """
    Cuenta operaciones como bloques contiguos
    donde la señal permanece activa.
    """
    sig = df[col].astype(int)
    block_id = (sig.diff().fillna(0) != 0).cumsum()
    return sum(1 for _, grp in df.groupby(block_id) if grp[col].iloc[0])

print("✅ Funciones de lógica operativa listas.")


✅ Funciones de lógica operativa listas.


---
## 📐 BLOQUE 3 — Construcción del Universo Operativo

Se construye el universo operativo a partir de las variables seleccionadas durante las etapas previas del proyecto, aplicando los mismos criterios metodológicos utilizados durante la validación.

• Universo A (Score DD+ATR): combina drawdown y volatilidad para identificar escenarios de oportunidad.

• Universo B (State Score): utiliza la información derivada del modelo de regímenes para construir señales operativas.

• Señales combinadas (OR / AND): unión e intersección de ambos universos bajo reglas de permanencia definidas.

La salida de este bloque corresponde al conjunto de señales y operaciones que será utilizado en las etapas posteriores de análisis y evaluación.


In [31]:
# =========================================================
# BLOQUE 3 — CONSTRUCCIÓN DEL UNIVERSO OPERATIVO
# =========================================================

def build_live_df(df_scores):
    """Construye el universo operativo replicando la lógica de la Parte 3.
    Devuelve (live_df, OPT_THR_VALUE). No recalcula state_score_opt: lo lee del CSV."""
    # ── Filtro de universo
    df_modern = df_scores[df_scores.index.year >= MODERN_YEAR_FILTER].copy()

    # ── Universo A: ranking drawdown + ATR
    df_modern["rank_dd"]      = df_modern["drawdown"].rank(pct=True, ascending=False)
    df_modern["rank_atr"]     = df_modern["atr_pct"].rank(pct=True, ascending=True)
    df_modern["score_dd_atr"] = 0.5 * df_modern["rank_dd"] + 0.5 * df_modern["rank_atr"]

    # ── Universo B: señal basada en state_score_opt ──
    OPT_THR_VALUE = df_modern["state_score_opt"].quantile(STATE_SCORE_PCT_THRESH)

    # ── Señales crudas ──
    dd_raw_signal  = df_modern["score_dd_atr"]    >= DD_RANK_THRESHOLD
    opt_raw_signal = df_modern["state_score_opt"] >= OPT_THR_VALUE

    # ── Señales con holding (rich, para paneles) ──
    dd_held, dd_fresh, dd_trig, dd_rem = build_holding_signal_rich(
        dd_raw_signal, DD_HOLD_WEEKS, df_modern.index)
    opt_held, opt_fresh, opt_trig, opt_rem = build_holding_signal_rich(
        opt_raw_signal.fillna(False), STATE_HOLD_WEEKS, df_modern.index)

    live_df = df_modern.copy()
    live_df["dd_raw"], live_df["dd_held"]   = dd_raw_signal, dd_held
    live_df["dd_fresh_trigger"]              = dd_fresh
    live_df["dd_trigger_date"]               = dd_trig
    live_df["dd_weeks_remaining"]            = dd_rem
    live_df["opt_raw"], live_df["opt_held"] = opt_raw_signal, opt_held
    live_df["opt_fresh_trigger"]             = opt_fresh
    live_df["opt_trigger_date"]              = opt_trig
    live_df["opt_weeks_remaining"]           = opt_rem

    live_df["and_active"] = live_df["dd_held"] & live_df["opt_held"]
    live_df["or_active"]  = live_df["dd_held"] | live_df["opt_held"]

    return live_df, OPT_THR_VALUE

print("✅ Función de construcción del universo operativo lista.")


✅ Función de construcción del universo operativo lista.


---
## 📘 BLOQUE 4 — Extracción de Operaciones OR y Paneles

Transforma el historial de señales del OR Champion en operaciones individuales. Cada operación conserva su fecha de entrada/salida, su retorno y la evolución de las variables internas mientras estuvo abierta.

Los resultados de este bloque generan dos estructuras principales:

• or_trades: resumen de cada operación identificada.

• or_panels: historial completo de variables y señales durante la vida de cada operación.

Estos objetos permiten analizar permanencia, evolución interna y comportamiento de las señales a nivel individual.

In [32]:
# =========================================================
# BLOQUE 4 — EXTRACCIÓN DE OPERACIONES OR + PANELES
# =========================================================

def extract_trades(df, active_col, internal_vars=None):
    """Extrae operaciones (bloques contiguos activos) + paneles temporales."""
    if internal_vars is None:
        internal_vars = [v for v in INTERNAL_VARS if v in df.columns]
    else:
        internal_vars = [v for v in internal_vars if v in df.columns]

    sig = df[active_col].astype(int)
    block_id = (sig.diff().fillna(0) != 0).cumsum()
    trades, panels = [], {}

    for _, grp in df.groupby(block_id):
        if not grp[active_col].iloc[0]:
            continue

        tid = len(trades)
        entry_date, exit_date = grp.index[0], grp.index[-1]
        n_weeks   = len(grp)
        ret_total = grp["returns"].add(1).prod() - 1
        years     = max(n_weeks / 52, 1 / 52)
        cagr      = (1 + ret_total) ** (1 / years) - 1

        info = {
            "trade_id": tid,
            "entry": entry_date,
            "exit": exit_date,
            "weeks": n_weeks,
            "return": ret_total,
            "cagr": cagr,
            "en_curso": exit_date == df.index[-1],
        }
        for v in internal_vars:
            info[f"{v}_entry"] = grp[v].iloc[0]
            info[f"{v}_exit"]  = grp[v].iloc[-1]
            if pd.api.types.is_numeric_dtype(grp[v]):
                info[f"{v}_mean"] = grp[v].mean()

        trades.append(info)
        keep_cols = internal_vars + [c for c in ["Close", "returns"] if c in grp.columns]
        panels[tid] = grp[keep_cols].copy()

    return pd.DataFrame(trades), panels

print("✅ Función de extracción de operaciones lista.")


✅ Función de extracción de operaciones lista.


---
## 🚦 BLOQUE 5 — Validación de Integridad

Antes de persistir nada, validamos la **integridad y consistencia** del estado.

La validación verifica:

• Presencia de variables operativas requeridas.

• Coherencia entre operaciones y paneles.

• Consistencia de los conteos reportados.

• Integridad de los metadatos generados.


In [33]:
# =========================================================
# BLOQUE 5 — VALIDACIÓN DEL ESTADO OPERATIVO
# =========================================================

class IntegrityError(RuntimeError):
    """Se lanza cuando el estado operativo no supera las validaciones."""
    pass


def validate_state_integrity(live_df, or_trades, or_panels, metadata):
    """Validaciones de integridad que NO dependen de referencia externa."""
    problems = []

    # 1) Columnas operativas presentes
    required = ["or_active", "and_active", "score_dd_atr",
                "state_score_opt", "dd_held", "opt_held"]
    missing = [c for c in required if c not in live_df.columns]
    if missing:
        problems.append(f"live_df no tiene columnas operativas: {missing}")

    # 2) live_df no vacío y ordenado temporalmente
    if len(live_df) == 0:
        problems.append("live_df está vacío.")
    elif not live_df.index.is_monotonic_increasing:
        problems.append("live_df no está ordenado temporalmente.")

    # 3) Coherencia entre operaciones y paneles
    if len(or_trades) != len(or_panels):
        problems.append(
            f"Inconsistencia entre operaciones ({len(or_trades)}) "
            f"y paneles ({len(or_panels)})."
        )

    # 4) Conteos coherentes con metadata
    n_or = count_events(live_df, "or_active")
    n_and = count_events(live_df, "and_active")

    if n_or != metadata.get("n_or_trades"):
        problems.append(
            f"n_or_trades en metadata ({metadata.get('n_or_trades')}) "
            f"≠ recontado ({n_or})."
        )

    if n_and != metadata.get("n_and_trades"):
        problems.append(
            f"n_and_trades en metadata ({metadata.get('n_and_trades')}) "
            f"≠ recontado ({n_and})."
        )

    if len(or_trades) != n_or:
        problems.append(
            f"len(or_trades)={len(or_trades)} ≠ eventos OR ({n_or})."
        )

    # 5) Umbral operativo válido
    if not np.isfinite(metadata.get("opt_thr_value", np.nan)):
        problems.append("opt_thr_value no es finito en metadata.")

    return problems, {"n_or": n_or, "n_and": n_and}


def gate_level2_reference(counts, manifest):
    """Compara los resultados actuales contra la referencia previamente registrada."""
    problems = []

    if manifest is None or manifest.get("status") not in (
        "canonical",
        "pending_bundle"
    ):
        return problems

    ref = manifest.get("counts", {})

    if ref.get("n_or") is not None and counts["n_or"] != ref["n_or"]:
        problems.append(
            f"OR actual ({counts['n_or']}) ≠ referencia ({ref['n_or']})."
        )

    if ref.get("n_and") is not None and counts["n_and"] != ref["n_and"]:
        problems.append(
            f"AND actual ({counts['n_and']}) ≠ referencia ({ref['n_and']})."
        )

    return problems


def gate_optional_expected(counts):
    """Validación opcional contra referencias EXPECTED_* definidas en el Bloque 0."""
    problems = []

    if EXPECTED_OR_TRADES is not None and counts["n_or"] != EXPECTED_OR_TRADES:
        problems.append(
            f"OR ({counts['n_or']}) ≠ EXPECTED_OR_TRADES ({EXPECTED_OR_TRADES})."
        )

    if EXPECTED_AND_TRADES is not None and counts["n_and"] != EXPECTED_AND_TRADES:
        problems.append(
            f"AND ({counts['n_and']}) ≠ EXPECTED_AND_TRADES ({EXPECTED_AND_TRADES})."
        )

    return problems


def run_gate(state, manifest):
    """Ejecuta todas las validaciones del estado operativo."""
    print("🔍 VALIDACIÓN DEL ESTADO OPERATIVO")
    print("=" * 60)

    l1, counts = validate_state_integrity(
        state["live_df"],
        state["or_trades"],
        state["or_panels"],
        state["metadata"]
    )

    l2 = gate_level2_reference(counts, manifest)
    le = gate_optional_expected(counts)

    all_problems = l1 + l2 + le

    print(f"  Operaciones OR  : {counts['n_or']}")
    print(f"  Operaciones AND : {counts['n_and']}")
    print(f"  Integridad interna          : {'OK' if not l1 else f'{len(l1)} problema(s)'}")
    print(f"  Consistencia histórica      : {'OK' if not l2 else f'{len(l2)} problema(s)'}")
    print(f"  Referencia EXPECTED_*       : {'OK' if not le else f'{len(le)} problema(s)'}")

    if all_problems:
        print("\n❌ VALIDACIÓN FALLIDA:")
        for p in all_problems:
            print(f"   • {p}")

        raise IntegrityError(
            "El estado operativo no supera las validaciones requeridas."
        )

    print("\n✅ Validación superada: el estado es íntegro y consistente.")
    return counts


print("✅ Validación del estado operativo lista.")

✅ Validación del estado operativo lista.


---
## 🧩 BLOQUE 6 — Verificación de Artefactos de Modelado

Este bloque verifica la disponibilidad de los artefactos de modelado
utilizados por las etapas operativas del proyecto.

Si los artefactos están disponibles, se registran dentro del estado
operativo generado. Si no están presentes, el proceso continúa
normalmente y deja constancia de su ausencia en los metadatos.


In [34]:
# =========================================================
# BLOQUE 6 — VERIFICACIÓN DE ARTEFACTOS DE MODELADO
# =========================================================

def check_bundle():
    """Verifica la disponibilidad de los artefactos de modelado.
    No realiza entrenamiento ni modificaciones;
    solo registra su presencia o ausencia.
    """
    if BUNDLE_PATH.exists():
        try:
            bundle = joblib.load(BUNDLE_PATH)
            keys = list(bundle.keys()) if isinstance(bundle, dict) else "objeto"
            return True, bundle, keys
        except Exception as e:
            return False, None, f"existe pero no se pudo cargar: {e}"
    return False, None, None


BUNDLE_AVAILABLE, bundle_obj, bundle_info = check_bundle()

if BUNDLE_AVAILABLE:
    print("✅ Bundle de modelado encontrado y cargado.")
    print(f"   Contenido: {bundle_info}")
    print("   El estado operativo podrá quedar COMPLETO (canonical).")
else:
    print("⚠️  Artefactos de modelado no disponibles,")
    print(" El proceso continuará utilizando únicamente los artefactos disponibles.")


⚠️  Artefactos de modelado no disponibles,
 El proceso continuará utilizando únicamente los artefactos disponibles.


---
## 💾 BLOQUE 7 — Persistencia de Resultados y Metadata

Aquí se decide qué hacer según el estado de la carpeta canónica:

| Estado encontrado | Comportamiento |
|-----------|--------|
| Existe MANIFEST íntegro y `FORCE_REBUILD=False` | **Cargar** y verificar hashes |
| Carpeta vacía/incompleta, o `FORCE_REBUILD=True` | **Construir** estado y persistir |

Los resultados generados por el notebook se almacenan junto con
su metadata descriptiva, permitiendo reproducibilidad,
trazabilidad y validación posterior.

Control de calidad:
los resultados deben superar las validaciones internas
antes de utilizarse como referencia.


In [35]:
# =========================================================
# BLOQUE 7 — Persistencia de Resultados y Metadata
# =========================================================

def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def file_hash(path, algo="sha256"):
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def all_state_artifacts_exist():
    return all(p.exists() for p in STATE_ARTIFACTS.values()) and MANIFEST_PATH.exists()


def load_manifest():
    if not MANIFEST_PATH.exists():
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def build_operative_state(df_scores):
    """
    Construye el dataset operativo utilizado por las etapas posteriores
    del proyecto a partir de los datasets validados.
    """
    print("⚙️ Generando el universo operativo...")

    live_df, opt_thr = build_live_df(df_scores)
    or_trades, or_panels = extract_trades(live_df, "or_active")

    metadata = {
        "universe_start": str(live_df.index.min().date()),
        "universe_end": str(live_df.index.max().date()),
        "modern_year": MODERN_YEAR_FILTER,
        "n_live_weeks": int(len(live_df)),
        "n_or_trades": int(count_events(live_df, "or_active")),
        "n_and_trades": int(count_events(live_df, "and_active")),
        "opt_thr_value": float(opt_thr),
        "source_part2_rows": int(len(df_scores)),
        "built_at": datetime.now().isoformat(),
    }

    print(f"   ✅ Universo operativo : {len(live_df):,} semanas")
    print(f"   ✅ Operaciones OR     : {metadata['n_or_trades']}")
    print(f"   ✅ Operaciones AND    : {metadata['n_and_trades']}")

    return {
        "live_df": live_df,
        "or_trades": or_trades,
        "or_panels": or_panels,
        "metadata": metadata,
    }


def make_state_id(metadata):
    seed = json.dumps(
        {
            "universe_end": metadata.get("universe_end"),
            "n_or_trades": metadata.get("n_or_trades"),
            "n_and_trades": metadata.get("n_and_trades"),
            "version": PIPELINE_VERSION,
        },
        sort_keys=True,
    )
    return hashlib.sha256(seed.encode()).hexdigest()[:12]


def persist_state(state):
    save_pickle(state["live_df"], STATE_ARTIFACTS["live_df"])
    save_pickle(state["or_trades"], STATE_ARTIFACTS["or_trades"])
    save_pickle(state["or_panels"], STATE_ARTIFACTS["or_panels"])
    save_pickle(state["metadata"], STATE_ARTIFACTS["metadata"])

    hashes = {
        name: file_hash(path)
        for name, path in STATE_ARTIFACTS.items()
    }

    if BUNDLE_AVAILABLE:
        hashes["model_artifacts"] = file_hash(BUNDLE_PATH)

    return hashes


def write_manifest(state, hashes, counts, bundle_present):
    md_ = state["metadata"]

    # Todo estado nuevo se registra inicialmente como "pending".
    # Posteriormente puede promoverse como referencia oficial.
    manifest = {
        "state_id": make_state_id(md_),
        "status": "pending",
        "bundle_present": bundle_present,
        "created_at": datetime.now().isoformat(),
        "canonized_at": None,
        "pipeline_version": PIPELINE_VERSION,
        "universe": {
            "start": md_.get("universe_start"),
            "end": md_.get("universe_end"),
            "modern_year": md_.get("modern_year"),
            "n_live_weeks": md_.get("n_live_weeks"),
        },
        "counts": {
            "n_or": counts["n_or"],
            "n_and": counts["n_and"],
        },
        "params": {
            "MODERN_YEAR_FILTER": MODERN_YEAR_FILTER,
            "DD_RANK_THRESHOLD": DD_RANK_THRESHOLD,
            "DD_HOLD_WEEKS": DD_HOLD_WEEKS,
            "STATE_SCORE_PCT_THRESH": STATE_SCORE_PCT_THRESH,
            "STATE_HOLD_WEEKS": STATE_HOLD_WEEKS,
        },
        "opt_thr_value": md_.get("opt_thr_value"),
        "hashes": hashes,
    }

    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    return manifest


def verify_hashes(manifest):
    mismatches = []

    targets = dict(STATE_ARTIFACTS)

    if manifest.get("bundle_present"):
        targets["model_artifacts"] = BUNDLE_PATH

    for name, path in targets.items():

        if not path.exists():
            mismatches.append(f"Falta el artefacto: {name}")
            continue

        signed = manifest.get("hashes", {}).get(name)

        if signed is not None and file_hash(path) != signed:
            mismatches.append(f"Hash inconsistente: {name}")

    return mismatches


# ---------------------------------------------------------
# Decisión: reutilizar resultados existentes o reconstruir
# ---------------------------------------------------------

_manifest = load_manifest()

if all_state_artifacts_exist() and not FORCE_REBUILD:

    print("📂 Estado operativo encontrado. Verificando integridad...")

    _mismatch = verify_hashes(_manifest)

    if _mismatch:

        print("   ❌ Se detectaron inconsistencias:")

        for m in _mismatch:
            print(f"      • {m}")

        raise ReconciliationError(
            "Los artefactos almacenados no coinciden con la metadata registrada."
        )

    state = {
        "live_df": pd.read_pickle(STATE_ARTIFACTS["live_df"]),
        "or_trades": pd.read_pickle(STATE_ARTIFACTS["or_trades"]),
        "or_panels": pd.read_pickle(STATE_ARTIFACTS["or_panels"]),
        "metadata": pd.read_pickle(STATE_ARTIFACTS["metadata"]),
    }

    STATE_WAS_BUILT = False

    _counts = run_gate(state, _manifest)

    print(f"   ✅ Estado cargado correctamente. state_id={_manifest['state_id']}")

else:

    if FORCE_REBUILD:
        print("🔁 FORCE_REBUILD=True → reconstruyendo resultados.")
    else:
        print("🆕 No se encontraron resultados persistidos. Se construirá un nuevo estado.")

    state = build_operative_state(df_scores)

    _counts = run_gate(state, _manifest)

    _hashes = persist_state(state)

    _manifest = write_manifest(
        state,
        _hashes,
        _counts,
        bundle_present=BUNDLE_AVAILABLE,
    )

    STATE_WAS_BUILT = True

    print(f"   ✅ Estado operativo generado y almacenado. state_id={_manifest['state_id']}")

    if not BUNDLE_AVAILABLE:
        print(
            "   ℹ️ El artefacto del modelo aún no está disponible; "
            "podrá incorporarse automáticamente cuando exista."
        )

📂 Estado operativo encontrado. Verificando integridad...
🔍 VALIDACIÓN DEL ESTADO OPERATIVO
  Operaciones OR  : 7
  Operaciones AND : 5
  Integridad interna          : OK
  Consistencia histórica      : OK
  Referencia EXPECTED_*       : OK

✅ Validación superada: el estado es íntegro y consistente.
   ✅ Estado cargado correctamente. state_id=b9fbfc029852


---
## 🏅 BLOQUE 8 — Validación y Aprobación del Estado

La construcción del universo operativo y su validación se mantienen como pasos separados.

Antes de utilizar los resultados como referencia del estudio, se recomienda revisar el reporte generado en el BLOQUE 9 para confirmar que los conteos, parámetros y cobertura temporal coinciden con los esperados.

Una vez revisado, basta con habilitar CANONIZE_PENDING_STATE=True en el BLOQUE 0 y ejecutar nuevamente esta etapa para registrar el estado como referencia oficial del proyecto.


In [36]:
# =========================================================
# BLOQUE 8 — VALIDACIÓN Y APROBACIÓN DEL ESTADO
# =========================================================

def effective_status(manifest):
    """Devuelve una descripción legible del estado registrado."""
    if manifest is None:
        return "No disponible"

    status = manifest.get("status", "?")

    if status == "canonical" and not manifest.get("bundle_present"):
        return "Referencia aprobada (modelo pendiente)"

    if status == "pending":
        return "Pendiente de revisión"

    return "Referencia aprobada"


if CANONIZE_PENDING_STATE:

    print("🏅 Iniciando proceso de aprobación del estado...")

    _m = load_manifest()

    if _m is None:
        raise RuntimeError(
            "No existe un estado registrado. Ejecute primero el proceso de construcción."
        )

    if _m.get("status") == "canonical":

        print(f"   ℹ️ El estado ya fue aprobado (state_id={_m['state_id']}).")

        # Si el bundle apareció posteriormente, incorporarlo.
        if not _m.get("bundle_present") and BUNDLE_AVAILABLE:

            _m["bundle_present"] = True
            _m["hashes"]["model_artifacts"] = file_hash(BUNDLE_PATH)

            with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
                json.dump(_m, f, indent=2, ensure_ascii=False)

            print("   ✅ Artefacto del modelo incorporado correctamente.")

    else:

        _mismatch = verify_hashes(_m)

        if _mismatch:

            print("   ❌ No es posible aprobar este estado:")
            for m in _mismatch:
                print(f"      • {m}")

            raise ReconciliationError(
                "Los artefactos almacenados no coinciden con la metadata registrada."
            )

        _m["status"] = "canonical"
        _m["canonized_at"] = datetime.now().isoformat()

        if BUNDLE_AVAILABLE and not _m.get("bundle_present"):

            _m["bundle_present"] = True
            _m["hashes"]["model_artifacts"] = file_hash(BUNDLE_PATH)

        with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
            json.dump(_m, f, indent=2, ensure_ascii=False)

        print("   ✅ Estado aprobado correctamente.")
        print(f"      state_id      : {_m['state_id']}")
        print(f"      Operaciones   : OR={_m['counts']['n_or']} | AND={_m['counts']['n_and']}")

        if _m["bundle_present"]:
            print("      Modelo        : disponible")
        else:
            print("      Modelo        : pendiente de incorporación")

        print("\n   ℹ️ Recuerde volver a establecer CANONIZE_PENDING_STATE=False.")

else:

    _m = load_manifest()

    print(f"ℹ️ Estado actual: {effective_status(_m)}")

    if _m and _m.get("status") == "pending":
        print(
            "   Revise el reporte del BLOQUE 9. "
            "Si los resultados son correctos, active "
            "CANONIZE_PENDING_STATE=True y ejecute nuevamente este bloque."
        )

ℹ️ Estado actual: Pendiente de revisión
   Revise el reporte del BLOQUE 9. Si los resultados son correctos, active CANONIZE_PENDING_STATE=True y ejecute nuevamente este bloque.


---
## 📋 BLOQUE 9 — Reporte Ejecutivo

Resumen del estado operativo disponible al finalizar la construcción. Se muestran los principales parámetros, el universo generado y los artefactos persistidos para facilitar la revisión del pipeline.


In [37]:
# =========================================================
# BLOQUE 9 — REPORTE Ejecutivo
# =========================================================

_m  = load_manifest()
md_ = state["metadata"]

print("=" * 64)
print("RESUMEN DEL ESTADO OPERATIVO")
print("=" * 64)
if _m:
    print(f"  state_id            : {_m['state_id']}")
    print(f"  estado     : {effective_status(_m)}")
    print(f"  modelo disponible   : {_m.get('bundle_present')}")
    print(f"  versión del pipeline : {_m['pipeline_version']}")
    print(f"  Fecha de construcción          : {_m['created_at']}")
    print(f"  Fecha de aprobación        : {_m['canonized_at']}")
print("  " + "-" * 60)
print(f"  Universo operativo  : {md_['universe_start']} → {md_['universe_end']}  "
      f"(año >= {md_['modern_year']})")
print(f"  Semanas live_df     : {md_['n_live_weeks']:,}")
print(f"  Sustrato Parte 2    : {md_['source_part2_rows']:,} filas")
print("  " + "-" * 60)
print(f"  Operaciones OR      : {md_['n_or_trades']}")
print(f"  Operaciones AND     : {md_['n_and_trades']}")
print(f"  OPT_THR_VALUE (P70) : {md_['opt_thr_value']:.4f}")
print("  " + "-" * 60)
print("  Artefactos generados:")
for name, path in STATE_ARTIFACTS.items():
    print(f"    {'✅' if path.exists() else '❌'} {path.name}")
print("  Modelo de inferencia:")
print(f"    {'✅' if BUNDLE_PATH.exists() else '⏳'} {BUNDLE_PATH.name}"
      f"{'' if BUNDLE_PATH.exists() else '   (pendiente — lo persistirá la Parte 1)'}")
print(f"    {'✅' if MANIFEST_PATH.exists() else '❌'} {MANIFEST_PATH.name}")
print("=" * 64)
if _m and not _m.get("bundle_present"):
    print("  ℹ️ Modelo de inferencia aún no disponible.")
    print("      El estado operativo fue construido correctamente y puede ser utilizado por las etapas posteriores del pipeline.")
    print("      Cuando el modelo de inferencia sea incorporado, quedará disponible automáticamente para la operación en vivo.")
print("  Las etapas posteriores del pipeline utilizan exclusivamente estos artefactos.")
print("=" * 64)


RESUMEN DEL ESTADO OPERATIVO
  state_id            : b9fbfc029852
  estado     : Pendiente de revisión
  modelo disponible   : False
  versión del pipeline : 3.5.0
  Fecha de construcción          : 2026-06-24T00:31:17.021798
  Fecha de aprobación        : None
  ------------------------------------------------------------
  Universo operativo  : 2010-01-15 → 2025-05-30  (año >= 2010)
  Semanas live_df     : 165
  Sustrato Parte 2    : 354 filas
  ------------------------------------------------------------
  Operaciones OR      : 7
  Operaciones AND     : 5
  OPT_THR_VALUE (P70) : 2.8000
  ------------------------------------------------------------
  Artefactos generados:
    ✅ live_df.pkl
    ✅ or_trades.pkl
    ✅ or_panels.pkl
    ✅ metadata.pkl
  Modelo de inferencia:
    ⏳ model_artifacts.joblib   (pendiente — lo persistirá la Parte 1)
    ✅ MANIFEST.json
  ℹ️ Modelo de inferencia aún no disponible.
      El estado operativo fue construido correctamente y puede ser utilizado por 